# Smart MCQ Solver — Fuzzy Lookup Only

**Roll No**: 24f1002384 | **Notebook**: `DL-24f1002384-notebook-t22026`

This notebook uses **only the improved fuzzy text-matching pipeline** to generate submissions.
No GPU needed. Runs in ~1 minute.

### Pipeline:
1. Strip preamble templates from prompts
2. Deduplicate training set (2000 rows → 419 unique questions)
3. Fuzzy match each test question to training using `token_set_ratio`
4. **Text-anchor resolution** — match answer TEXT, not letter, to handle option reordering
5. Export `submission.csv`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz'], check=True)
print('Ready')

In [ ]:
import os, re
import pandas as pd
from tqdm.auto import tqdm
from rapidfuzz import fuzz, process

DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'
CHOICES  = list('ABCDE')

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f'Train: {train_df.shape} | Test: {test_df.shape}')

In [ ]:
# Strip instruction preambles to expose the core question text
PREAMBLES = [
    r'^Pick the best possible answer:\s*',
    r'^Select the most accurate option:\s*',
    r'^Identify the correct statement:\s*',
    r'^Choose the correct answer:\s*',
    r'^Determine the correct option:\s*',
    r'^Which of the following is correct\?\s*',
    r'\s*among the listed options\.?\s*$',
    r'\s*from the following choices\.?\s*$',
    r'\s*carefully\.?\s*$',
]

def strip_preambles(text: str) -> str:
    text = str(text).strip()
    for p in PREAMBLES:
        text = re.sub(p, '', text, flags=re.IGNORECASE).strip()
    return text

train_df['core'] = train_df['prompt'].apply(strip_preambles)
test_df['core']  = test_df['prompt'].apply(strip_preambles)

print(f'Unique core questions in train: {train_df["core"].nunique()}')
print(f'Before: {train_df["prompt"].iloc[0][:80]}...')
print(f'After:  {train_df["core"].iloc[0]}')

In [ ]:
# Deduplicate train — 419 unique questions instead of 2000 rows
train_deduped       = train_df.drop_duplicates(subset='core', keep='first').reset_index(drop=True)
train_deduped_cores = train_deduped['core'].tolist()

def resolve_answer_for_test_row(train_row, test_row):
    """
    Text-Anchor Resolution: finds which test option contains the correct answer text.
    Handles option reordering between train and test rows.
    
    1. Try exact text match first (100% certainty)
    2. Fuzzy token_set_ratio fallback (handles minor rewording)
    3. Letter-copy as last resort (if fuzzy confidence < 85)
    """
    correct_label = train_row['answer']
    correct_text  = str(train_row[correct_label]).strip()

    # Priority 1: Exact text match
    for c in CHOICES:
        if str(test_row[c]).strip() == correct_text:
            return c

    # Priority 2: Fuzzy match on the answer text
    best_letter = correct_label
    best_score  = 0
    for c in CHOICES:
        sim = fuzz.token_set_ratio(correct_text, str(test_row[c]).strip())
        if sim > best_score:
            best_score  = sim
            best_letter = c

    if best_score >= 85:
        return best_letter
    return correct_label  # fallback to letter-copy

lookup_hard = {}
lookup_soft = {}

for _, test_row in tqdm(test_df.iterrows(), total=len(test_df), desc='Fuzzy lookup'):
    res = process.extractOne(test_row['core'], train_deduped_cores, scorer=fuzz.token_set_ratio)
    if res:
        score, match_idx = res[1], res[2]
        matched_train_row = train_deduped.iloc[match_idx]
        resolved_label = resolve_answer_for_test_row(matched_train_row, test_row)

        if score >= 95:
            lookup_hard[int(test_row['id'])] = resolved_label
        elif score >= 80:
            lookup_soft[int(test_row['id'])] = resolved_label

print(f'Hard matches (>=95): {len(lookup_hard)} / {len(test_df)}')
print(f'Soft matches (80-94): {len(lookup_soft)} / {len(test_df)}')
print(f'No match: {len(test_df) - len(lookup_hard) - len(lookup_soft)}')

In [ ]:
# Generate final predictions — fuzzy only, no models
final_predictions = {}

for _, row in test_df.iterrows():
    tid     = str(int(row['id']))
    test_id = int(row['id'])

    if test_id in lookup_hard:
        correct_ans = lookup_hard[test_id]
        remaining   = [c for c in CHOICES if c != correct_ans]
        final_predictions[tid] = [correct_ans] + remaining[:2]
    elif test_id in lookup_soft:
        correct_ans = lookup_soft[test_id]
        remaining   = [c for c in CHOICES if c != correct_ans]
        final_predictions[tid] = [correct_ans] + remaining[:2]
    else:
        final_predictions[tid] = ['A', 'B', 'C']

print(f'Predictions generated: {len(final_predictions)}')

In [ ]:
# Verify and export submission.csv
errors = []
test_ids = {str(int(i)) for i in test_df['id']}

if set(final_predictions.keys()) != test_ids:
    errors.append(f'ID mismatch: expected {len(test_ids)}, got {len(final_predictions)}')

for tid, preds in final_predictions.items():
    if len(preds) != 3:
        errors.append(f'ID {tid}: expected 3 predictions, got {len(preds)}')
    if len(set(preds)) != 3:
        errors.append(f'ID {tid}: duplicate predictions {preds}')
    if not all(p in CHOICES for p in preds):
        errors.append(f'ID {tid}: invalid choice {preds}')

if not errors:
    print('All checks passed!')
    rows = [
        {'ID': idx, 'Prediction': ' '.join(final_predictions[str(int(idx))])}
        for idx in sorted(test_df['id'].tolist(), key=int)
    ]
    sub_df = pd.DataFrame(rows)
    sub_df.to_csv('submission.csv', index=False)
    print(f'Saved submission.csv ({len(sub_df)} rows)')
    print(sub_df.head(10))
else:
    print('Errors:', errors[:5])

In [ ]:
# Quick distribution check
if os.path.exists('submission.csv'):
    sub_df = pd.read_csv('submission.csv')
    top1 = sub_df['Prediction'].str.split().str[0]
    print('Top-1 prediction distribution:')
    print(top1.value_counts().sort_index())
    print(f'\nHard lookup: {len(lookup_hard)}')
    print(f'Soft lookup: {len(lookup_soft)}')
    print(f'No match:    {len(test_df) - len(lookup_hard) - len(lookup_soft)}')